# Distributed state vector simulations on multiple GPUs (advanced)

In the notebook "5_Multiple_GPU_simulations.ipynb", you learned how to use CUDA-Q and Braket Hybrid Jobs to parallelize the simulation of a batch of observables and circuits over multiple GPUs, where each GPU simulates a single QPU. For workloads with larger qubit counts, however, it may be necessary to distribute a single state vector simulation across multiple GPUs so that multiple GPUs together simulate a single QPU.

In this notebook, you will learn how to use CUDA-Q and Braket Hybrid Jobs to tackle this.

We start with necessary imports that are used in the examples below.

In [1]:
from braket.aws import AwsQuantumJob, AwsSession
from braket.jobs.config import InstanceConfig
from braket.jobs.image_uris import Framework, retrieve_image

We use the CUDA-Q hybrid jobs container provided by Braket, which contains CUDA-Q, cuStateVec, and CUDA-aware OpenMPI required for distributing state vector simulations across multiple GPUs.

In [2]:
image_uri = retrieve_image(Framework.CUDAQ, AwsSession().region)
print(f"Image: {image_uri}")

Image: 292282985366.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-cudaq-jobs:latest


## Distributed state vector simulations

The `nvidia` target with the `mgpu` option distributes a single state vector simulation across multiple GPUs. This enables simulations for circuits with higher qubit counts — up to 34 qubits with 4 GPUs.

To use `mgpu`, we need MPI to coordinate the GPUs. We enable this by setting `sagemaker_mpi_enabled=true` and specifying the number of MPI processes per host (one per GPU). The `ml.g6.12xlarge` instance has 4 NVIDIA L4 GPUs.

Unlike the previous notebooks that use the `@hybrid_job` decorator, here we write the algorithm as a standalone Python file and submit it with `AwsQuantumJob.create()`. This gives us direct control over the MPI hyperparameters. The `%%writefile` cell magic writes the cell contents to a local file that is then uploaded as the job's source module.

**Important notes for multi-GPU scripts:**
- All MPI ranks must operate on the same Hamiltonian (mgpu is a collective operation). Here we use a fixed seed for `SpinOperator.random`; alternatively, one could generate the Hamiltonian on one rank and broadcast it to the others with `MPI.bcast`.
- Only one process saves results (to avoid redundant S3 writes) — here we choose rank 0.

In [3]:
%%writefile mgpu_algorithm.py
"""Distributed state vector simulation across multiple GPUs using CUDA-Q mgpu."""

import os
import cudaq
from braket.jobs import save_job_result


def main():
    rank = int(os.getenv("OMPI_COMM_WORLD_RANK", "0"))
    size = int(os.getenv("OMPI_COMM_WORLD_SIZE", "1"))
    print(f"[Rank {rank}/{size}] num_available_gpus={cudaq.num_available_gpus()}")

    cudaq.set_target("nvidia", option="mgpu")

    n_qubits = int(os.getenv("SM_HP_N_QUBITS", "25"))
    n_shots = int(os.getenv("SM_HP_N_SHOTS", "10000000"))

    @cudaq.kernel
    def ghz():
        qubits = cudaq.qvector(n_qubits)
        h(qubits[0])
        for q in range(1, n_qubits):
            cx(qubits[0], qubits[q])

    # All ranks must use the same Hamiltonian (mgpu is a collective operation)
    hamiltonian = cudaq.SpinOperator.random(n_qubits, 10, seed=42)

    result = cudaq.observe(ghz, hamiltonian, shots_count=n_shots)
    print(f"[Rank {rank}] expectation={result.expectation()}")

    # Only rank 0 saves the result (including GPU memory to confirm distribution)
    if rank == 0:
        import subprocess
        mem = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader"],
            capture_output=True, text=True,
        )
        save_job_result({
            "expectation": result.expectation(),
            "gpu_memory": mem.stdout.strip(),
        })


if __name__ == "__main__":
    main()

Overwriting mgpu_algorithm.py


In [4]:
job = AwsQuantumJob.create(
    device="local:nvidia/nvidia-mgpu",
    source_module="mgpu_algorithm.py",
    entry_point="mgpu_algorithm:main",
    instance_config=InstanceConfig(instanceType="ml.g6.12xlarge", instanceCount=1),
    image_uri=image_uri,
    hyperparameters={
        "sagemaker_mpi_enabled": "true",
        "sagemaker_mpi_num_of_processes_per_host": "4",
        "n_qubits": "28",
        "n_shots": "10000000",
    },
    wait_until_complete=False,
)
print("Job ARN:", job.arn)

Job ARN: arn:aws:braket:us-east-1:656977514066:job/2dcad83b-37d8-4093-a52f-8afb69d24116


In [5]:
result = job.result()
print(f"Expectation value (approximate): {result['expectation']}")
print(f"GPU memory per device (rank 0 node only):\n{result['gpu_memory']}")

Expectation value (approximate): 0.0010958000000000077
GPU memory per device (rank 0 node only):
0, 1246 MiB
1, 798 MiB
2, 798 MiB
3, 798 MiB


## Multi-node distributed simulation

You can also distribute the state vector across GPUs on multiple nodes. The example below uses 2 nodes with 1 GPU each, giving 2 GPUs total. This allows simulating circuits that don't fit in a single node's GPU's memory.

In [6]:
multi_node_job = AwsQuantumJob.create(
    device="local:nvidia/nvidia-mgpu",
    source_module="mgpu_algorithm.py",
    entry_point="mgpu_algorithm:main",
    instance_config=InstanceConfig(instanceType="ml.g6.xlarge", instanceCount=2),
    image_uri=image_uri,
    hyperparameters={
        "sagemaker_mpi_enabled": "true",
        "sagemaker_mpi_num_of_processes_per_host": "1",
        "n_qubits": "28",
        "n_shots": "10000000",
    },
    wait_until_complete=False,
)
print("Job ARN:", multi_node_job.arn)

Job ARN: arn:aws:braket:us-east-1:656977514066:job/9fb07c0c-0e35-4409-bb8b-ead84e56ba55


In [7]:
result = multi_node_job.result()
print(f"Expectation value (approximate): {result['expectation']}")
print(f"GPU memory per device (rank 0 node only):\n{result['gpu_memory']}")

Expectation value (approximate): -0.0009536000000000389
GPU memory per device (rank 0 node only):
0, 1758 MiB


## Summary

This notebook shows how to distribute a single state vector simulation across multiple GPUs using CUDA-Q's `nvidia-mgpu` backend and MPI. Key points:

- Set `sagemaker_mpi_enabled=true` and `sagemaker_mpi_num_of_processes_per_host` to the number of GPUs per node
- All ranks must operate on the same Hamiltonian (use a fixed seed or broadcast)
- One process saves results to avoid redundant S3 writes
- Works both single-node (multiple GPUs) and multi-node (one or more GPUs per node)
- Maximum 5 nodes per Braket Hybrid Job